# 04 — RFM Segmentation

## Purpose
This notebook builds customer RFM scores from the validated PostgreSQL warehouse.

The goal is to:
- pull customer-level RFM base data from PostgreSQL
- score recency, frequency, and monetary behavior
- assign business-friendly segment labels
- write the final RFM segment table back into PostgreSQL

This notebook uses the validated `fact_sales` and `dim_customer` tables rather than old flat-file derived outputs.

In [1]:
import pandas as pd
import sys
from sqlalchemy import text

sys.path.insert(0, '..')
from src.utils.db_loader import get_engine

engine = get_engine()

## Step 1 — Pull RFM base from PostgreSQL

We use the validated warehouse tables to build the RFM base directly from PostgreSQL.

This avoids depending on deprecated flat files and keeps segmentation aligned with the current warehouse.

In [2]:
rfm_sql = """
WITH customer_base AS (
    SELECT
        dc.customer_id,
        MAX(fs.order_date) AS last_order_date,
        COUNT(DISTINCT fs.invoice) AS frequency,
        ROUND(SUM(fs.revenue), 2) AS monetary
    FROM fact_sales fs
    JOIN dim_customer dc
        ON fs.customer_key = dc.customer_key
    WHERE fs.customer_key <> 0
    GROUP BY dc.customer_id
)
SELECT
    customer_id,
    (SELECT MAX(order_date) FROM fact_sales) - last_order_date AS recency,
    frequency,
    monetary
FROM customer_base
ORDER BY monetary DESC
"""

rfm = pd.read_sql(rfm_sql, engine)

print("RFM base shape:", rfm.shape)
display(rfm.head())
display(rfm.describe())

RFM base shape: (5878, 4)


,customer_id,recency,frequency,monetary
0,18102,0,145,580987.04
1,14646,1,151,528602.52
2,14156,9,156,313437.62
3,14911,1,398,291420.81
4,17450,8,51,244784.25


,customer_id,recency,frequency,monetary
count,5878.000000,5878.000000,5878.000000,5878.000000
mean,15315.313542,200.866791,6.289384,2955.904092
std,1715.572666,209.353961,13.009406,14440.852686
min,12346.000000,0.000000,1.000000,2.950000
25%,13833.250000,25.000000,1.000000,342.280000
50%,15314.500000,95.000000,3.000000,867.740000
75%,16797.750000,379.000000,7.000000,2248.305000
max,18287.000000,738.000000,398.000000,580987.040000


## Step 1 result — RFM base extracted from PostgreSQL

The validated warehouse returned **5,878 customer-level RFM rows**.

### Base structure
Each row represents one identifiable customer with:
- `recency` → days since last purchase relative to the latest warehouse sales date
- `frequency` → number of distinct invoices
- `monetary` → total customer revenue

### Key distribution observations
- Average recency: **200.87 days**
- Median recency: **95 days**
- Average frequency: **6.29 invoices**
- Median frequency: **3 invoices**
- Average monetary value: **2,955.90**
- Median monetary value: **867.74**
- Maximum monetary value: **580,987.04**

### Business implication
Customer value is highly uneven. A relatively small number of customers generate very large monetary totals, while the median customer contributes much less. This confirms that segmentation is necessary rather than relying on simple averages.

## Step 2 — Score recency, frequency, and monetary dimensions

We assign quintile-based scores from 1 to 5.

### Scoring logic
- **R = 5** means the customer purchased most recently
- **F = 5** means the customer purchased most frequently
- **M = 5** means the customer generated the highest monetary value

In [3]:
rfm["r_score"] = pd.qcut(
    rfm["recency"],
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

rfm["f_score"] = pd.qcut(
    rfm["frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm["m_score"] = pd.qcut(
    rfm["monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm["rfm_score"] = (
    rfm["r_score"].astype(str)
    + rfm["f_score"].astype(str)
    + rfm["m_score"].astype(str)
)

rfm["rfm_total"] = rfm["r_score"] + rfm["f_score"] + rfm["m_score"]

display(rfm.head())

,customer_id,recency,frequency,monetary,r_score,f_score,m_score,rfm_score,rfm_total
0,18102,0,145,580987.04,5,5,5,555,15
1,14646,1,151,528602.52,5,5,5,555,15
2,14156,9,156,313437.62,5,5,5,555,15
3,14911,1,398,291420.81,5,5,5,555,15
4,17450,8,51,244784.25,5,5,5,555,15


## Step 2 result — RFM scores assigned

Each customer was scored into quintiles across recency, frequency, and monetary dimensions.

### Score interpretation
- **R = 5** → most recent customers
- **F = 5** → most frequent customers
- **M = 5** → highest-value customers

The top customers in the dataset received the strongest combined scores, for example:
- customer `18102` → `555`
- customer `14646` → `555`
- customer `14156` → `555`
- customer `14911` → `555`
- customer `17450` → `555`

### Business implication
The scoring logic successfully distinguishes the highest-value and most recently active customers from the broader customer base, creating a strong foundation for customer segmentation.

## Step 3 — Assign segment labels

These segment rules create business-readable customer groups from the scored RFM values.

The labels are intentionally simple and interpretable for dashboarding and stakeholder review.

In [4]:
def segment_customer(row):
    r = row["r_score"]
    f = row["f_score"]
    m = row["m_score"]

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif r >= 3 and f >= 3:
        return "Loyal Customers"
    elif r >= 4 and f <= 2:
        return "New Customers"
    elif r <= 2 and m >= 4:
        return "At Risk - High Value"
    elif r <= 2 and f >= 3:
        return "At Risk - Frequent"
    elif r == 1 and f == 1:
        return "Lost"
    else:
        return "Potential Loyalists"

rfm["segment"] = rfm.apply(segment_customer, axis=1)

print("Segment counts:")
display(rfm["segment"].value_counts())

Segment counts:


segment
Loyal Customers         1423
Potential Loyalists     1348
Champions               1268
At Risk - Frequent       516
Lost                     476
New Customers            453
At Risk - High Value     394
Name: count, dtype: int64

## Step 3 result — customer segment labels assigned

The scored customers were grouped into interpretable business segments.

### Segment distribution
- **Loyal Customers** → **1,423**
- **Potential Loyalists** → **1,348**
- **Champions** → **1,268**
- **At Risk - Frequent** → **516**
- **Lost** → **476**
- **New Customers** → **453**
- **At Risk - High Value** → **394**

### Interpretation
The customer base is strongest in three large groups:
- **Champions**
- **Loyal Customers**
- **Potential Loyalists**

At the same time, the presence of:
- **At Risk - Frequent**
- **At Risk - High Value**
- **Lost**

shows that a meaningful portion of customers may require retention attention.

### Business implication
This segmentation can support:
- retention campaigns
- high-value customer monitoring
- reactivation strategies
- differentiated dashboard views by customer segment

## Step 4 — Prepare final PostgreSQL load format

The final dataframe is aligned to the `ml_rfm_segments` table schema already created in PostgreSQL.

In [5]:
rfm_final = rfm[
    [
        "customer_id",
        "recency",
        "frequency",
        "monetary",
        "r_score",
        "f_score",
        "m_score",
        "rfm_score",
        "rfm_total",
        "segment",
    ]
].copy()

display(rfm_final.head())
print("Final RFM rows:", len(rfm_final))

,customer_id,recency,frequency,monetary,r_score,f_score,m_score,rfm_score,rfm_total,segment
0,18102,0,145,580987.04,5,5,5,555,15,Champions
1,14646,1,151,528602.52,5,5,5,555,15,Champions
2,14156,9,156,313437.62,5,5,5,555,15,Champions
3,14911,1,398,291420.81,5,5,5,555,15,Champions
4,17450,8,51,244784.25,5,5,5,555,15,Champions


Final RFM rows: 5878


## Step 4 result — final PostgreSQL load dataset prepared

The final RFM segmentation dataset contains **5,878 rows**, aligned to the PostgreSQL `ml_rfm_segments` schema.

### Final columns prepared
- `customer_id`
- `recency`
- `frequency`
- `monetary`
- `r_score`
- `f_score`
- `m_score`
- `rfm_score`
- `rfm_total`
- `segment`

### Business implication
The segmentation output is now in a warehouse-ready format and can be used directly by:
- SQL queries
- Power BI
- Tableau
- later AI summary generation

## Step 5 — Write RFM segments back to PostgreSQL

We do not replace the table schema.  
Instead, we clear the existing rows and append the fresh scored output.

In [6]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE ml_rfm_segments"))

rfm_final.to_sql(
    "ml_rfm_segments",
    engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=2000
)

print("RFM segments written to PostgreSQL successfully.")

RFM segments written to PostgreSQL successfully.


## Step 5 result — RFM segments written back to PostgreSQL

The fresh RFM segmentation output was written successfully into the PostgreSQL warehouse without replacing the table schema.

### Why this matters
This keeps the segmentation layer:
- reproducible
- queryable from SQL
- available for dashboard tools
- ready for future AI-based daily analytics and reporting

## Step 6 — Quick validation

Confirm that the RFM segment table was loaded successfully.

In [7]:
validation_sql = """
SELECT
    COUNT(*) AS row_count
FROM ml_rfm_segments
"""

segment_check_sql = """
SELECT
    segment,
    COUNT(*) AS customer_count
FROM ml_rfm_segments
GROUP BY segment
ORDER BY customer_count DESC
"""

validation_df = pd.read_sql(validation_sql, engine)
segment_check_df = pd.read_sql(segment_check_sql, engine)

display(validation_df)
display(segment_check_df)

,row_count
0,5878


,segment,customer_count
0,Loyal Customers,1423
1,Potential Loyalists,1348
2,Champions,1268
3,At Risk - Frequent,516
4,Lost,476
5,New Customers,453
6,At Risk - High Value,394


## Step 6 result — PostgreSQL validation passed

The `ml_rfm_segments` table contains **5,878 rows**, matching the expected number of identifiable customers from the validated warehouse.

### Validated segment counts
- **Loyal Customers** → **1,423**
- **Potential Loyalists** → **1,348**
- **Champions** → **1,268**
- **At Risk - Frequent** → **516**
- **Lost** → **476**
- **New Customers** → **453**
- **At Risk - High Value** → **394**

### Validation conclusion
The RFM segmentation pipeline is now fully integrated into PostgreSQL and consistent with the earlier warehouse benchmarks.

## RFM segmentation conclusion

This notebook successfully built a warehouse-based RFM segmentation layer from validated PostgreSQL sales data.

### What was achieved
- Pulled customer-level RFM base directly from PostgreSQL
- Scored recency, frequency, and monetary behavior using quintiles
- Assigned business-readable customer segments
- Wrote the final segmentation table back into PostgreSQL
- Validated the final row count and segment distribution

### Final result
The warehouse now contains **5,878 segmented customers** across seven business-readable groups:
- Champions
- Loyal Customers
- Potential Loyalists
- New Customers
- At Risk - High Value
- At Risk - Frequent
- Lost

### Why this matters
This segmentation layer strengthens the project in three ways:
1. it enables customer-focused dashboards and BI reporting
2. it supports future retention and revenue-risk analysis
3. it provides structured customer intelligence for later AI-driven daily reporting